# 🏭 굴뚝 탐지 (Chimney Detection) - YOLOv8

Kompsat-3/3A 위성 이미지를 활용한 굴뚝 위치 탐지 프로젝트

- **목표**: 위성 이미지에서 굴뚝을 탐지하여 바운딩 박스로 위치 예측
- **모델**: YOLOv8 (Object Detection)
- **평가지표**: mAP@IoU=0.5

## 📋 1. 환경 설정 및 패키지 설치

In [ ]:
# 필요 패키지 설치
!pip install ultralytics==8.0.196
!pip install opencv-python==4.8.1.78
!pip install Pillow==10.0.0
!pip install PyYAML==6.0.1
!pip install tqdm==4.66.1
!pip install pandas==2.0.3
!pip install seaborn==0.12.2

import os
import json
import shutil
from pathlib import Path
import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import torch
from ultralytics import YOLO

print("✅ 패키지 설치 완료!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 📁 2. Google Drive 마운트 및 데이터 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 데이터 경로 설정
drive_root = "/content/drive/MyDrive"
img_dir = f"{drive_root}/TS_KS"  # 이미지 폴더
label_dir = f"{drive_root}/TL_KS_BBOX"  # 라벨 폴더

# 데이터 확인
print(f"📸 이미지 폴더: {img_dir}")
print(f"📋 라벨 폴더: {label_dir}")

if os.path.exists(img_dir) and os.path.exists(label_dir):
    img_count = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    label_count = len([f for f in os.listdir(label_dir) if f.endswith('.json')])
    print(f"✅ 이미지 파일: {img_count}개")
    print(f"✅ 라벨 파일: {label_count}개")
    
    # 샘플 파일 확인
    sample_imgs = [f for f in os.listdir(img_dir) if f.endswith('.jpg')][:3]
    sample_labels = [f for f in os.listdir(label_dir) if f.endswith('.json')][:3]
    print(f"\n📄 샘플 이미지: {sample_imgs}")
    print(f"📄 샘플 라벨: {sample_labels}")
else:
    print("❌ 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")

## 🔧 3. 데이터 전처리 및 YOLO 형식 변환

In [ ]:
class DataPreparator:
    def __init__(self, img_dir, label_dir, output_dir="yolo_dataset"):
        self.img_dir = Path(img_dir)
        self.label_dir = Path(label_dir)
        self.output_dir = Path(output_dir)
        
    def create_yolo_structure(self):
        """YOLO 데이터셋 구조 생성"""
        directories = [
            self.output_dir / "images" / "train",
            self.output_dir / "images" / "val", 
            self.output_dir / "labels" / "train",
            self.output_dir / "labels" / "val"
        ]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
            
    def convert_bbox_to_yolo(self, bbox_info, img_width, img_height):
        """바운딩 박스를 YOLO 형식으로 변환"""
        x = bbox_info["x"]
        y = bbox_info["y"] 
        width = bbox_info["width"]
        height = bbox_info["height"]
        
        # YOLO 형식: center_x, center_y, width, height (모두 정규화)
        center_x = (x + width / 2) / img_width
        center_y = (y + height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height
        
        return center_x, center_y, norm_width, norm_height
    
    def process_json_labels(self, json_file_path, img_path):
        """JSON 라벨 파일을 처리하여 YOLO 형식으로 변환"""
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            img = cv2.imread(str(img_path))
            if img is None:
                return None
                
            img_height, img_width = img.shape[:2]
            
            yolo_labels = []
            
            # JSON 구조 파싱 (여러 가능성 고려)
            annotations = []
            if "annotations" in data:
                annotations = data["annotations"]
            elif "shapes" in data:
                annotations = data["shapes"]
            elif isinstance(data, list):
                annotations = data
            
            for annotation in annotations:
                bbox = None
                
                # shape_attributes 형식
                if "shape_attributes" in annotation:
                    bbox = annotation["shape_attributes"]
                # 직접 좌표가 있는 경우
                elif "x" in annotation and "y" in annotation:
                    bbox = annotation
                
                if bbox and bbox.get("name") == "rect" or ("x" in bbox and "y" in bbox):
                    center_x, center_y, norm_width, norm_height = self.convert_bbox_to_yolo(
                        bbox, img_width, img_height
                    )
                    # class_id = 0 (굴뚝 클래스)
                    yolo_labels.append(f"0 {center_x:.6f} {center_y:.6f} {norm_width:.6f} {norm_height:.6f}")
            
            return yolo_labels
            
        except Exception as e:
            print(f"Error processing {json_file_path}: {e}")
            return None
    
    def prepare_dataset(self, train_ratio=0.8):
        """전체 데이터셋 준비 (훈련/검증 분할)"""
        self.create_yolo_structure()
        
        # 모든 이미지 파일 리스트
        img_files = list(self.img_dir.glob("*.jpg"))
        print(f"총 이미지 파일: {len(img_files)}개")
        
        # 훈련/검증 분할
        import random
        random.seed(42)
        random.shuffle(img_files)
        
        train_count = int(len(img_files) * train_ratio)
        train_files = img_files[:train_count]
        val_files = img_files[train_count:]
        
        print(f"훈련 데이터: {len(train_files)}개")
        print(f"검증 데이터: {len(val_files)}개")
        
        # 훈련/검증 데이터 처리
        self.process_split(train_files, "train")
        self.process_split(val_files, "val")
        
        # YAML 설정 파일 생성
        self.create_yaml_config()
        
    def process_split(self, img_files, split):
        """훈련/검증 분할 처리"""
        processed = 0
        skipped = 0
        
        for img_file in tqdm(img_files, desc=f"Processing {split} data"):
            # 라벨 파일 찾기
            label_file = self.label_dir / f"{img_file.stem}.json"
            
            if not label_file.exists():
                skipped += 1
                continue
                
            # 이미지 복사
            dst_img = self.output_dir / "images" / split / img_file.name
            shutil.copy2(img_file, dst_img)
            
            # 라벨 변환 및 저장
            yolo_labels = self.process_json_labels(label_file, img_file)
            if yolo_labels:
                dst_label = self.output_dir / "labels" / split / f"{img_file.stem}.txt"
                with open(dst_label, 'w') as f:
                    f.write('\n'.join(yolo_labels))
                    
                processed += 1
            else:
                skipped += 1
                
        print(f"✅ {split}: {processed}개 처리 완료, {skipped}개 건너뜀")
        
    def create_yaml_config(self):
        """YOLO 설정 YAML 파일 생성"""
        config = {
            'path': str(self.output_dir.absolute()),
            'train': 'images/train',
            'val': 'images/val',
            'nc': 1,  # 클래스 수 (굴뚝 1개)
            'names': ['chimney']  # 클래스 이름
        }
        
        with open(self.output_dir / "dataset.yaml", 'w') as f:
            yaml.dump(config, f, default_flow_style=False)
            
        print(f"✅ 데이터셋 설정 파일 생성: {self.output_dir / 'dataset.yaml'}")

# 데이터 전처리 실행
print("🔧 데이터 전처리 시작...")
preparator = DataPreparator(img_dir, label_dir)
preparator.prepare_dataset(train_ratio=0.8)
print("✅ 데이터 전처리 완료!")

## 👀 4. 데이터 샘플 확인

In [ ]:
# 변환된 데이터 확인
dataset_path = Path("yolo_dataset")

# 데이터셋 구조 확인
for split in ['train', 'val']:
    img_count = len(list((dataset_path / "images" / split).glob("*.jpg")))
    label_count = len(list((dataset_path / "labels" / split).glob("*.txt")))
    print(f"📊 {split}: {img_count}개 이미지, {label_count}개 라벨")

# 샘플 이미지와 라벨 시각화
sample_img = list((dataset_path / "images" / "train").glob("*.jpg"))[0]
sample_label = dataset_path / "labels" / "train" / f"{sample_img.stem}.txt"

# 이미지 읽기
img = cv2.imread(str(sample_img))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_height, img_width = img.shape[:2]

# 라벨 읽기
if sample_label.exists():
    with open(sample_label, 'r') as f:
        labels = f.read().strip().split('\n')
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    
    # 바운딩 박스 그리기
    for label in labels:
        if label.strip():
            parts = label.split()
            class_id, cx, cy, w, h = map(float, parts)
            
            # YOLO 형식을 픽셀 좌표로 변환
            x1 = int((cx - w/2) * img_width)
            y1 = int((cy - h/2) * img_height)
            x2 = int((cx + w/2) * img_width)
            y2 = int((cy + h/2) * img_height)
            
            # 바운딩 박스 그리기
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                               fill=False, color='red', linewidth=2)
            plt.gca().add_patch(rect)
            plt.text(x1, y1-5, f'Chimney', color='red', fontsize=10, fontweight='bold')
    
    plt.title(f'샘플 데이터 확인: {sample_img.name}')
    plt.axis('off')
    plt.show()
    
    print(f"✅ 샘플 이미지에서 {len(labels)}개 굴뚝 발견")
else:
    print("❌ 라벨 파일을 찾을 수 없습니다.")

## 🚀 5. YOLOv8 모델 훈련

In [ ]:
class ChimneyDetector:
    def __init__(self, model_name="yolov8n.pt", data_config="yolo_dataset/dataset.yaml"):
        self.model_name = model_name
        self.data_config = data_config
        self.model = None
        
    def initialize_model(self):
        """YOLO 모델 초기화"""
        print(f"🤖 YOLOv8 모델 초기화: {self.model_name}")
        self.model = YOLO(self.model_name)
        
        # GPU 사용 가능 시 GPU 사용
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🔧 사용 디바이스: {device}")
        
    def train_model(self, epochs=50, img_size=640, batch_size=16):
        """모델 훈련"""
        if self.model is None:
            self.initialize_model()
            
        print("🎯 모델 훈련 시작...")
        
        # 훈련 파라미터 (Colab 환경에 최적화)
        training_args = {
            'data': self.data_config,
            'epochs': epochs,
            'imgsz': img_size,
            'batch': batch_size,
            'name': 'chimney_detection',
            'save_period': 10,  # 10 에포크마다 저장
            'patience': 15,     # Early stopping patience
            'save': True,
            'plots': True,
            'val': True,
            'device': 0 if torch.cuda.is_available() else 'cpu'
        }
        
        # 훈련 실행
        results = self.model.train(**training_args)
        
        print("✅ 훈련 완료!")
        return results
    
    def validate_model(self, model_path="runs/detect/chimney_detection/weights/best.pt"):
        """모델 검증"""
        print("📊 모델 검증 중...")
        
        # 최고 성능 모델 로드
        if Path(model_path).exists():
            self.model = YOLO(model_path)
        
        # 검증 실행
        results = self.model.val(data=self.data_config)
        
        # mAP@0.5 결과 출력
        map50 = results.box.map50
        print(f"🎯 mAP@IoU=0.5: {map50:.4f}")
        
        return results

# 모델 훈련 실행
detector = ChimneyDetector()

# Colab 환경에 맞는 파라미터로 훈련 (빠른 실행을 위해 에포크 수 줄임)
print("🚀 굴뚝 탐지 모델 훈련 시작...")
batch_size = 8 if torch.cuda.is_available() else 4  # GPU 메모리에 따라 조정
training_results = detector.train_model(epochs=50, img_size=640, batch_size=batch_size)

print("🎉 훈련 완료! 검증 시작...")
validation_results = detector.validate_model()

## 📈 6. 훈련 결과 확인

In [ ]:
# 훈련 곡선 시각화
results_path = Path("runs/detect/chimney_detection")
if (results_path / "results.png").exists():
    from IPython.display import Image, display
    print("📊 훈련 곡선:")
    display(Image(str(results_path / "results.png")))
else:
    print("❌ 훈련 결과 이미지를 찾을 수 없습니다.")

# 혼동 행렬 확인
if (results_path / "confusion_matrix.png").exists():
    print("\n🎯 혼동 행렬:")
    display(Image(str(results_path / "confusion_matrix.png")))

# 검증 예측 결과 확인
val_imgs = list(results_path.glob("val_batch*.jpg"))
if val_imgs:
    print("\n🔍 검증 예측 결과:")
    for val_img in val_imgs[:2]:  # 처음 2개만 표시
        display(Image(str(val_img)))

## 🎯 7. 정확한 mAP@IoU=0.5 계산

In [ ]:
class ChimneyEvaluator:
    def __init__(self, model_path):
        self.model_path = model_path
        self.model = YOLO(model_path) if Path(model_path).exists() else None
        
    def calculate_iou(self, box1, box2):
        """두 바운딩 박스 간의 IoU 계산"""
        x1_max = max(box1[0], box2[0])
        y1_max = max(box1[1], box2[1]) 
        x2_min = min(box1[2], box2[2])
        y2_min = min(box1[3], box2[3])
        
        if x2_min <= x1_max or y2_min <= y1_max:
            return 0.0
            
        intersection = (x2_min - x1_max) * (y2_min - y1_max)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0.0
    
    def evaluate_model(self, test_images_dir="yolo_dataset/images/val", 
                      test_labels_dir="yolo_dataset/labels/val", conf_threshold=0.25):
        """모델 평가 실행"""
        if self.model is None:
            print("❌ 모델을 로드할 수 없습니다.")
            return None
            
        print("📊 정확한 mAP@IoU=0.5 계산 중...")
        
        # Ground truth 로드
        ground_truths = {}
        label_files = list(Path(test_labels_dir).glob("*.txt"))
        
        for label_file in label_files:
            img_name = label_file.stem + ".jpg"
            boxes = []
            
            with open(label_file, 'r') as f:
                for line in f:
                    if line.strip():
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            class_id, cx, cy, w, h = map(float, parts[:5])
                            boxes.append([cx, cy, w, h])
            
            ground_truths[img_name] = boxes
        
        # 예측 수행
        predictions = {}
        test_images = list(Path(test_images_dir).glob("*.jpg"))
        
        correct_predictions = 0
        total_predictions = 0
        total_ground_truths = 0
        
        for img_path in tqdm(test_images, desc="예측 진행 중"):
            img_name = img_path.name
            
            # 이미지 크기 가져오기
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_height, img_width = img.shape[:2]
            
            # 모델 예측
            results = self.model(img_path, conf=conf_threshold, verbose=False)
            
            pred_boxes = []
            pred_scores = []
            
            for result in results:
                if result.boxes is not None:
                    boxes = result.boxes.xyxy.cpu().numpy()
                    scores = result.boxes.conf.cpu().numpy()
                    
                    for box, score in zip(boxes, scores):
                        pred_boxes.append(box.tolist())
                        pred_scores.append(score)
            
            total_predictions += len(pred_boxes)
            
            # Ground truth 박스를 xyxy 형식으로 변환
            if img_name in ground_truths:
                gt_boxes = ground_truths[img_name]
                total_ground_truths += len(gt_boxes)
                
                gt_xyxy = []
                for gt_box in gt_boxes:
                    cx, cy, w, h = gt_box
                    x1 = (cx - w/2) * img_width
                    y1 = (cy - h/2) * img_height
                    x2 = (cx + w/2) * img_width
                    y2 = (cy + h/2) * img_height
                    gt_xyxy.append([x1, y1, x2, y2])
                
                # IoU 계산으로 정확한 예측 수 계산
                for pred_box in pred_boxes:
                    for gt_box in gt_xyxy:
                        iou = self.calculate_iou(pred_box, gt_box)
                        if iou >= 0.5:
                            correct_predictions += 1
                            break
        
        # 정확도 계산
        precision = correct_predictions / total_predictions if total_predictions > 0 else 0
        recall = correct_predictions / total_ground_truths if total_ground_truths > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\n🎯 평가 결과:")
        print(f"정확한 예측: {correct_predictions}")
        print(f"총 예측: {total_predictions}")
        print(f"총 실제 굴뚝: {total_ground_truths}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1_score:.4f}")
        print(f"\n🏆 근사 mAP@IoU=0.5: {f1_score:.4f}")
        
        return {
            'correct_predictions': correct_predictions,
            'total_predictions': total_predictions,
            'total_ground_truths': total_ground_truths,
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score,
            'approximate_map': f1_score
        }

# 정확한 평가 실행
model_path = "runs/detect/chimney_detection/weights/best.pt"
if Path(model_path).exists():
    evaluator = ChimneyEvaluator(model_path)
    evaluation_results = evaluator.evaluate_model()
    
    # 결과 저장
    with open("evaluation_results.json", "w") as f:
        json.dump(evaluation_results, f, indent=2)
    print("\n💾 평가 결과가 evaluation_results.json에 저장되었습니다.")
else:
    print("❌ 훈련된 모델을 찾을 수 없습니다.")

## 🔍 8. 샘플 예측 결과 확인

In [ ]:
# 훈련된 모델로 샘플 예측
model_path = "runs/detect/chimney_detection/weights/best.pt"
if Path(model_path).exists():
    model = YOLO(model_path)
    
    # 검증 이미지에서 샘플 선택
    val_images = list(Path("yolo_dataset/images/val").glob("*.jpg"))
    sample_images = val_images[:3] if len(val_images) >= 3 else val_images
    
    fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))
    if len(sample_images) == 1:
        axes = [axes]
    
    for idx, img_path in enumerate(sample_images):
        # 예측 수행
        results = model(img_path, conf=0.25)
        
        # 이미지 로드
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(img_rgb)
        
        chimney_count = 0
        for result in results:
            if result.boxes is not None:
                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                
                for box, score in zip(boxes, scores):
                    x1, y1, x2, y2 = box
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                       fill=False, color='red', linewidth=2)
                    axes[idx].add_patch(rect)
                    axes[idx].text(x1, y1-5, f'Chimney: {score:.2f}', 
                           color='red', fontsize=8, fontweight='bold')
                    chimney_count += 1
        
        axes[idx].set_title(f'{img_path.name}\n{chimney_count}개 굴뚝 탐지')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ {len(sample_images)}개 샘플 이미지 예측 완료!")
else:
    print("❌ 훈련된 모델을 찾을 수 없습니다.")

## 📋 9. 결과 요약

In [ ]:
print("🎉 굴뚝 탐지 프로젝트 완료!")
print("="*50)

# 데이터셋 정보
dataset_path = Path("yolo_dataset")
if dataset_path.exists():
    train_imgs = len(list((dataset_path / "images" / "train").glob("*.jpg")))
    val_imgs = len(list((dataset_path / "images" / "val").glob("*.jpg")))
    print(f"📊 데이터셋:")
    print(f"   - 훈련: {train_imgs}개 이미지")
    print(f"   - 검증: {val_imgs}개 이미지")

# 모델 정보
model_path = Path("runs/detect/chimney_detection/weights/best.pt")
if model_path.exists():
    print(f"\n🤖 모델:")
    print(f"   - 최고 성능 모델: {model_path}")
    print(f"   - 모델 크기: {model_path.stat().st_size / (1024*1024):.1f} MB")

# 평가 결과
eval_file = Path("evaluation_results.json")
if eval_file.exists():
    with open(eval_file, 'r') as f:
        eval_data = json.load(f)
    
    print(f"\n🎯 평가 결과:")
    print(f"   - mAP@IoU=0.5: {eval_data.get('approximate_map', 0):.4f}")
    print(f"   - Precision: {eval_data.get('precision', 0):.4f}")
    print(f"   - Recall: {eval_data.get('recall', 0):.4f}")
    print(f"   - F1-Score: {eval_data.get('f1_score', 0):.4f}")

# 결과 파일들
print(f"\n📁 생성된 파일들:")
result_files = [
    "yolo_dataset/",
    "runs/detect/chimney_detection/", 
    "evaluation_results.json"
]

for file_path in result_files:
    if Path(file_path).exists():
        print(f"   ✅ {file_path}")
    else:
        print(f"   ❌ {file_path}")

print(f"\n🚀 다음 단계:")
print(f"   1. mAP@IoU=0.5 점수 확인")
print(f"   2. 하이퍼파라미터 튜닝 (필요시)")
print(f"   3. 더 많은 에포크로 재훈련 (필요시)")
print(f"   4. 실제 데이터로 테스트")